# 04 — Multi-Agent Memory Management: LangGraph, LangChain, and MCP Tools

**Audience:** developers building collaborative multi-agent systems using frameworks like **LangGraph** or **LangChain** who want to coordinate agent memory, sessions, and private-vs-shared knowledge bases.

This notebook demonstrates how to utilize `memtomem`'s Multi-Agent MCP tools to:

- Register distinct agents with separate private scopes.
- Manage episodic sessions to track agent task lifecycles.
- Use working memory (scratchpad) within a session.
- Share specific key decisions or contracts from private agent space to the `shared` namespace.
- Perform multi-agent search that retrieves relevant private and shared memory context.
- Integrate these components into a unified **LangGraph** state workflow.

## Setup & Prerequisites

We need **memtomem** (with the ONNX extra for local embedding model), **langchain**, **langgraph**, and **nest-asyncio** (to run async loops inside Jupyter).

```bash
uv pip install "memtomem[onnx]" langchain langgraph nest-asyncio jupyter ipykernel
```

In [ ]:
import sys

try:
    import langchain_core
    import langgraph
    import nest_asyncio
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"Missing dependency: '{e.name}'. Please install it and restart the kernel:\n"
        '  uv pip install "memtomem[onnx]" langchain langgraph nest-asyncio'
    ) from e

print("ONNX embedding backend, LangChain, and LangGraph are available.")

## Step 1 — Set up in-process memtomem components

We run the services against a temporary SQLite database and local folder so that no files land in your real `~/.memtomem/` database.

In [ ]:
import json
import os
import tempfile
from pathlib import Path
from types import SimpleNamespace

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    create_components,
    close_components,
)
from memtomem.server.context import AppContext

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables.
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "onnx"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "bge-small-en-v1.5"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "384"

# 3. Temporarily disable configuration loading from the user's home directory
import memtomem.config as _cfg
_orig_config_d_loader = _cfg.load_config_d
_orig_loader = _cfg.load_config_overrides
_cfg.load_config_d = lambda c, *args, **kwargs: None
_cfg.load_config_overrides = lambda c: None

config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

components = await create_components(config)
app = AppContext.from_components(components)

# Wrap AppContext in an MCP-like context format expected by tool handlers
ctx = SimpleNamespace(request_context=SimpleNamespace(lifespan_context=app))

print(f"memtomem components ready. db={db_path}")

## Step 2 — Wrap memtomem MCP tools for LangChain

We wrap the native `memtomem` MCP tools into standard LangChain/LangGraph tools. This lets agents interact with the memory backend using plain Python functions.

In [ ]:
from langchain_core.tools import tool

# Import tool handlers from memtomem's server package
from memtomem.server.tools.multi_agent import mem_agent_register, mem_agent_search, mem_agent_share
from memtomem.server.tools.session import mem_session_start, mem_session_end
from memtomem.server.tools.scratch import mem_scratch_set, mem_scratch_get
from memtomem.server.tools.memory_crud import mem_add

@tool
async def agent_register(agent_id: str, description: str = None) -> str:
    """Register an agent in the multi-agent memory system."""
    return await mem_agent_register(agent_id=agent_id, description=description, ctx=ctx)

@tool
async def session_start(agent_id: str, title: str = None) -> str:
    """Start an episodic memory session for a specific agent."""
    return await mem_session_start(agent_id=agent_id, title=title, ctx=ctx)

@tool
async def session_end(summary: str = None) -> str:
    """End the current episodic session and clean up working memory."""
    return await mem_session_end(summary=summary, ctx=ctx)

@tool
async def scratch_set(key: str, value: str) -> str:
    """Save temporary working memory / scratchpad for the current session."""
    return await mem_scratch_set(key=key, value=value, ctx=ctx)

@tool
async def scratch_get(key: str) -> str:
    """Retrieve temporary working memory for the current session."""
    return await mem_scratch_get(key=key, ctx=ctx)

@tool
async def agent_share(chunk_id: str, target: str = "shared") -> str:
    """Copy a memory chunk into another namespace (e.g. shared)."""
    return await mem_agent_share(chunk_id=chunk_id, target=target, ctx=ctx)

@tool
async def agent_search(query: str, agent_id: str = None, include_shared: bool = True) -> str:
    """Search memories with agent-scope awareness (private + shared namespaces)."""
    return await mem_agent_search(
        query=query, agent_id=agent_id, include_shared=include_shared, output_format="structured", ctx=ctx
    )

@tool
async def add_memory(content: str, tags: str = None) -> str:
    """Save long-term memory under the current active agent's scope."""
    tag_list = [t.strip() for t in tags.split(",")] if tags else []
    res = await mem_add(content=content, tags=tag_list, ctx=ctx)
    return res

print("Wrapped memtomem MCP tools successfully.")

## Step 3 — Multi-Agent Collaboration Scenario

We model two agents working on an API design:
1. **Developer Agent**: Creates the authentication API specification and registers it.
2. **Reviewer Agent**: Searches for the specification to perform standard checks.

In [ ]:
# 1. Register agents
print(await agent_register.ainvoke({"agent_id": "developer", "description": "Core developer writing APIs"}))
print(await agent_register.ainvoke({"agent_id": "reviewer", "description": "Reviews APIs for compliance"}))

### Developer Agent writes the API Contract and Shares it

In [ ]:
# 2. Developer starts an episodic session
print(await session_start.ainvoke({"agent_id": "developer", "title": "Develop Authentication Endpoint"}))

# 3. Developer stores local/temporary workflow variables in the scratchpad
await scratch_set.ainvoke({"key": "git_branch", "value": "feature/auth-endpoints"})

# 4. Developer saves the final API contract in long-term memory
# (automatically routed to the private `agent-runtime:developer` namespace)
await add_memory.ainvoke({
    "content": "Auth login API contract: POST /auth/login returns JWT response `{'token': '<string>'}`.",
    "tags": "contract,api"
})

# 5. Developer searches private memory in structured form to get the chunk ID
search_res = await agent_search.ainvoke({"query": "POST /auth/login", "agent_id": "developer"})
print("Developer Private Search Results:\n", search_res)

# 6. Share this contract with the 'shared' namespace so other agents can access it
search_payload = json.loads(search_res)
chunk_ids = [row["chunk_id"] for row in search_payload["results"]]
if chunk_ids:
    chunk_id = chunk_ids[0]
    print(f"\nSharing memory chunk {chunk_id} to shared namespace...")
    print(await agent_share.ainvoke({"chunk_id": chunk_id, "target": "shared"}))

# 7. End Developer session
print(await session_end.ainvoke({"summary": "Finished Auth login handler specification"}))

### Reviewer Agent checks the contract in the Shared Space

In [ ]:
# 1. Reviewer starts an episodic session
print(await session_start.ainvoke({"agent_id": "reviewer", "title": "Review Auth Endpoint Contract"}))

# 2. Reviewer searches for the authentication contract
# (Searches private 'agent-runtime:reviewer' + 'shared' namespaces)
reviewer_search = await agent_search.ainvoke({"query": "auth login contract", "agent_id": "reviewer"})
print("Reviewer Search Results:\n", reviewer_search)

assert "Auth login API contract" in reviewer_search, "Reviewer failed to find shared memory"
print("\n✓ Reviewer successfully read the shared API contract!")

# 3. Reviewer records review approval
await add_memory.ainvoke({
    "content": "Reviewer verdict: auth login API contract is APPROVED.",
    "tags": "verdict,api"
})

# 4. End Reviewer session
print(await session_end.ainvoke({"summary": "Reviewed and approved auth login contract"}))

## Step 4 — Orchestrate with LangGraph

We design a workflow where Agents act as nodes in a graph. The Developer Agent writes a spec, shares it, and the Reviewer Agent automatically retrieves it and makes a decision.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# Define state layout
class MultiAgentState(TypedDict):
    shared_keys: list[str]

# Developer agent node
async def run_developer_node(state: MultiAgentState) -> dict:
    print("\n--- [Developer Agent Running] ---")
    await mem_session_start(agent_id="developer", title="Logout Feature Spec", ctx=ctx)
    
    # Save logout contract
    await mem_add(content="Logout contract: POST /auth/logout accepts JWT in headers. Returns HTTP 204.", tags=["logout"], ctx=ctx)
    
    # Locate the chunk ID
    res = await mem_agent_search(query="Logout contract", agent_id="developer", output_format="structured", ctx=ctx)
    chunk_ids = [row["chunk_id"] for row in json.loads(res)["results"]]
    
    shared = []
    if chunk_ids:
        chunk_id = chunk_ids[0]
        await mem_agent_share(chunk_id=chunk_id, target="shared", ctx=ctx)
        shared.append(chunk_id)
        print(f"Shared logout contract chunk: {chunk_id}")
        
    await mem_session_end(summary="Specified and shared logout contract", ctx=ctx)
    return {"shared_keys": shared}

# Reviewer agent node
async def run_reviewer_node(state: MultiAgentState) -> dict:
    print("\n--- [Reviewer Agent Running] ---")
    await mem_session_start(agent_id="reviewer", title="Review Logout Feature", ctx=ctx)
    
    # Retrieve shared memory
    res = await mem_agent_search(query="logout contract", agent_id="reviewer", ctx=ctx)
    print("Reviewer retrieved memory:\n", res)
    
    decision = "APPROVED" if "POST /auth/logout" in res else "REJECTED"
    print(f"Reviewer decision: {decision}")
    
    await mem_add(content=f"Reviewer verdict: Logout spec is {decision}.", tags=["review"], ctx=ctx)
    await mem_session_end(summary=f"Logout review complete: {decision}", ctx=ctx)
    return {}

# Compile the state graph
builder = StateGraph(MultiAgentState)
builder.add_node("developer", run_developer_node)
builder.add_node("reviewer", run_reviewer_node)
builder.set_entry_point("developer")
builder.add_edge("developer", "reviewer")
builder.add_edge("reviewer", END)

graph = builder.compile()
print("LangGraph Multi-Agent Memory Workflow compiled successfully.")

### Run the Graph

In [ ]:
nest_asyncio.apply()

initial_state = {
    "shared_keys": []
}

await graph.ainvoke(initial_state)
print("\n✓ Multi-Agent LangGraph run finished successfully!")

## Connecting with Deep Agents Code (dcode)

Deep Agents Code (`dcode`) is an open-source terminal coding agent built on the Deep Agents SDK. It supports loading external tools via the Model Context Protocol (MCP).

To configure `dcode` to use `memtomem` as an MCP server, add a `memtomem` entry to your `.mcp.json` config file at one of the following locations:

1. **User-level (applies to all projects):** `~/.deepagents/.mcp.json`
2. **Project-level (hidden):** `<project-dir>/.deepagents/.mcp.json`
3. **Project-level (standard):** `<project-dir>/.mcp.json` (compatible with Claude Code)

### Example `.mcp.json` configuration:

```json
{
  "mcpServers": {
    "memtomem": {
      "command": "memtomem-server"
    }
  }
}
```

Once configured, starting a `dcode` session will auto-discover the `memtomem` MCP tools. The agent will then be able to register itself, search memories, share notes, and manage episodic session summaries dynamically.

## Teardown & Clean up

Shut down the active components and delete the temporary directory.

In [ ]:
await close_components(components)
tmp.cleanup()
_cfg.load_config_d = _orig_config_d_loader
_cfg.load_config_overrides = _orig_loader
print("Cleanup complete.")